# Domain E: Functional Connectivity and Circuit Interactions

This notebook addresses five questions:
- **E1:** Do noise correlations reveal cell-type-specific connectivity motifs?
- **E2:** Can transfer entropy / Granger causality reveal directed functional interactions?
- **E3:** How does population coupling relate to cell type and spatial position?
- **E4:** Does spontaneous activity structure during grey-screen epochs reveal cell-type-specific network dynamics? *(Zarr data)*
- **E5:** Do catch-trial responses reveal cell-type-specific expectation signals? *(Zarr data)*

**Data:** Zarr multimodal stores with ΔF/F traces (cells × trials), 3D coordinates, cell-type labels, gene expression, spontaneous activity, catch trials, and GLM data.

**Note:** E2 (Granger causality) requires continuous time-series data. The current dataset stores trial-level responses. Sections requiring continuous ΔF/F traces are marked `# ⚠️ REQUIRES: continuous time-series data`.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, mannwhitneyu
from scipy.spatial.distance import cdist, pdist, squareform
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import xcorr_pair, pairwise_granger

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR, include_genes=True) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# Gene columns
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias', 'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]

print(f"Total cells: {len(obs)}, Trials: {X.shape[1]}, Genes: {len(GENE_COLS)}")

## E1: Noise Correlations and Cell-Type-Specific Connectivity Motifs

Compute trial-by-trial noise correlations (residual correlations after removing stimulus-driven component). Build subclass × subclass average noise correlation matrix. Relate to spatial distance and gene expression similarity.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E1.1  Compute noise correlations per mouse
# ══════════════════════════════════════════════════════════════════════

# For each unique stimulus condition, subtract the trial-averaged response,
# then compute pairwise Pearson correlation of the residuals.

# Process one mouse at a time to keep matrices manageable
mouse_ids = obs['mouse_id'].unique()
contrast_blocks = var['stim_block'].isin([0.0, 2.0])

noise_corr_per_mouse = {}
signal_corr_per_mouse = {}

for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_X = X[m_mask]
    n_m = m_mask.sum()
    
    # Collect residuals across all stimulus conditions
    residuals_list = []
    signal_vectors = []
    
    for ori in orientations:
        for contrast in [0.05, 0.1, 0.2, 0.4, 0.8]:
            mask = contrast_blocks & (var['orientation'] == ori) & (var['contrast'] == contrast)
            trial_idx = np.where(mask.values)[0]
            if len(trial_idx) >= 3:
                trial_resp = m_X[:, trial_idx]  # cells x trials
                mean_resp = np.nanmean(trial_resp, axis=1)
                signal_vectors.append(mean_resp)
                residuals = trial_resp - mean_resp[:, np.newaxis]
                residuals_list.append(residuals)
    
    if residuals_list:
        all_residuals = np.hstack(residuals_list)  # cells x total_residual_trials
        nc = np.corrcoef(all_residuals)
        noise_corr_per_mouse[mouse] = nc
        
        signal_mat = np.column_stack(signal_vectors)  # cells x conditions
        sc = np.corrcoef(signal_mat)
        signal_corr_per_mouse[mouse] = sc
        
        print(f"Mouse {mouse}: {n_m} cells, {all_residuals.shape[1]} residual trials, "
              f"mean noise corr = {nc[np.triu_indices(n_m, k=1)].mean():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E1.2  Subclass × Subclass noise correlation matrix
# ══════════════════════════════════════════════════════════════════════

n_sc = len(present_subclasses)
nc_subclass_matrix = np.zeros((n_sc, n_sc))
nc_subclass_counts = np.zeros((n_sc, n_sc))

for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_subclass = obs['subclass_name'].values[m_mask]
    nc = noise_corr_per_mouse.get(mouse)
    if nc is None:
        continue
    
    n_m = m_mask.sum()
    for si, sc1 in enumerate(present_subclasses):
        for sj, sc2 in enumerate(present_subclasses):
            mask1 = m_subclass == sc1
            mask2 = m_subclass == sc2
            if mask1.sum() == 0 or mask2.sum() == 0:
                continue
            
            nc_sub = nc[np.ix_(np.where(mask1)[0], np.where(mask2)[0])]
            if si == sj:
                # Same subclass: use upper triangle only
                triu = np.triu_indices(nc_sub.shape[0], k=1)
                if len(triu[0]) > 0:
                    vals = nc_sub[triu]
                    nc_subclass_matrix[si, sj] += np.nansum(vals)
                    nc_subclass_counts[si, sj] += np.sum(~np.isnan(vals))
            else:
                vals = nc_sub.flatten()
                nc_subclass_matrix[si, sj] += np.nansum(vals)
                nc_subclass_counts[si, sj] += np.sum(~np.isnan(vals))

nc_subclass_avg = nc_subclass_matrix / np.maximum(nc_subclass_counts, 1)

# ── Visualization: Subclass × Subclass noise correlation heatmap ──
short_labels = [SUBCLASS_SHORT[s] for s in present_subclasses]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sns.heatmap(nc_subclass_avg, annot=True, fmt='.4f', xticklabels=short_labels,
            yticklabels=short_labels, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('E1: Mean Noise Correlation (Subclass × Subclass)', fontweight='bold')

# Same-type vs cross-type comparison
ax = axes[1]
same_type_nc = []
cross_type_nc = []
for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_subclass = obs['subclass_name'].values[m_mask]
    nc = noise_corr_per_mouse.get(mouse)
    if nc is None:
        continue
    n_m = m_mask.sum()
    triu_i, triu_j = np.triu_indices(n_m, k=1)
    for ii, jj in zip(triu_i, triu_j):
        val = nc[ii, jj]
        if np.isnan(val):
            continue
        if m_subclass[ii] == m_subclass[jj]:
            same_type_nc.append(val)
        else:
            cross_type_nc.append(val)

data_plot = pd.DataFrame({
    'Noise Correlation': same_type_nc + cross_type_nc,
    'Pair Type': ['Same subclass']*len(same_type_nc) + ['Different subclass']*len(cross_type_nc)
})
sns.violinplot(data=data_plot, x='Pair Type', y='Noise Correlation', cut=0, inner='box',
               palette=['red', 'blue'], ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.4)
ax.set_title('E1: Same-Type vs Cross-Type Noise Correlation', fontweight='bold')
stat, p = mannwhitneyu(same_type_nc, cross_type_nc, alternative='two-sided')
ax.text(0.5, 0.95, f'Mann-Whitney p={p:.2e}', transform=ax.transAxes, ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Same-type noise corr: mean={np.mean(same_type_nc):.4f}, n={len(same_type_nc)}")
print(f"Cross-type noise corr: mean={np.mean(cross_type_nc):.4f}, n={len(cross_type_nc)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E1.3  Noise correlations vs distance and gene-expression similarity
#       Mantel test: NC matrix ~ distance matrix + gene similarity
# ══════════════════════════════════════════════════════════════════════

# Use largest mouse for detailed analysis
mouse_pick = mouse_ids[np.argmax([np.sum(obs['mouse_id'].values == m) for m in mouse_ids])]
m_mask = obs['mouse_id'].values == mouse_pick
m_coords = coords[m_mask]
nc = noise_corr_per_mouse[mouse_pick]
n_m = m_mask.sum()

triu_i, triu_j = np.triu_indices(n_m, k=1)
pair_nc = nc[triu_i, triu_j]
pair_dist = squareform(pdist(m_coords))[triu_i, triu_j]

# Gene expression similarity (correlation)
gene_expr = obs.loc[m_mask, GENE_COLS].values.astype(float)
gene_expr = np.nan_to_num(gene_expr, nan=0.0)
nonzero = np.std(gene_expr, axis=0) > 1e-6
gene_corr_mat = np.corrcoef(gene_expr[:, nonzero])
pair_gene_sim = gene_corr_mat[triu_i, triu_j]

# ── Noise correlation vs distance ──
distance_edges = np.arange(0, 400, 25)
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# NC vs distance
ax = axes[0]
binned_nc = []
for b in range(len(distance_edges)-1):
    bm = (pair_dist >= distance_edges[b]) & (pair_dist < distance_edges[b+1]) & ~np.isnan(pair_nc)
    if bm.sum() > 10:
        binned_nc.append(np.mean(pair_nc[bm]))
    else:
        binned_nc.append(np.nan)
ax.plot(distance_centers, binned_nc, 'ko-', linewidth=2)
ax.set_xlabel('Pairwise Distance (µm)')
ax.set_ylabel('Mean Noise Correlation')
ax.set_title('E1: Noise Corr vs Distance', fontweight='bold')
ax.axhline(0, color='gray', ls='--', alpha=0.4)

# NC vs gene expression similarity
ax = axes[1]
gene_bins = np.percentile(pair_gene_sim[~np.isnan(pair_gene_sim)], np.arange(0, 101, 10))
gene_centers = (gene_bins[:-1] + gene_bins[1:]) / 2
binned_nc_gene = []
for b in range(len(gene_bins)-1):
    bm = (pair_gene_sim >= gene_bins[b]) & (pair_gene_sim < gene_bins[b+1]) & ~np.isnan(pair_nc)
    if bm.sum() > 10:
        binned_nc_gene.append(np.mean(pair_nc[bm]))
    else:
        binned_nc_gene.append(np.nan)
ax.plot(gene_centers, binned_nc_gene, 'ro-', linewidth=2)
ax.set_xlabel('Gene Expression Similarity (Pearson r)')
ax.set_ylabel('Mean Noise Correlation')
ax.set_title('E1: Noise Corr vs Gene Similarity', fontweight='bold')

# Partial correlation: NC ~ distance + gene_sim + cell-type match
ax = axes[2]
m_subclass = obs['subclass_name'].values[m_mask]
pair_same = (m_subclass[triu_i] == m_subclass[triu_j]).astype(float)

# Simple multiple regression
from numpy.linalg import lstsq
valid = ~np.isnan(pair_nc) & ~np.isnan(pair_gene_sim) & ~np.isnan(pair_dist)
A = np.column_stack([pair_dist[valid], pair_gene_sim[valid], pair_same[valid], np.ones(valid.sum())])
Y = pair_nc[valid]
beta, residuals, _, _ = lstsq(A, Y, rcond=None)
print(f"Multiple regression: NC ~ distance + gene_sim + same_type")
print(f"  β_distance = {beta[0]:.6f}")
print(f"  β_gene_sim = {beta[1]:.4f}")
print(f"  β_same_type = {beta[2]:.4f}")
print(f"  β_intercept = {beta[3]:.4f}")
ss_res = np.sum((Y - A @ beta)**2)
ss_tot = np.sum((Y - np.mean(Y))**2)
r2 = 1 - ss_res / ss_tot
print(f"  R² = {r2:.4f}")

# Bar plot of standardized coefficients
A_std = (A[:, :3] - A[:, :3].mean(axis=0)) / (A[:, :3].std(axis=0) + 1e-8)
A_std_full = np.column_stack([A_std, np.ones(len(A_std))])
beta_std, _, _, _ = lstsq(A_std_full, (Y - Y.mean()) / Y.std(), rcond=None)
ax.bar(['Distance', 'Gene Sim', 'Same Type'], beta_std[:3],
       color=['steelblue', 'salmon', 'green'])
ax.set_ylabel('Standardized β')
ax.set_title(f'E1: Predictors of NC (R²={r2:.3f})', fontweight='bold')
ax.axhline(0, color='k', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()

## E2: Directed Functional Connectivity (Cross-Correlations + Granger Causality)

Using the 10 Hz ΔF/F data from zarr stores, we now have proper time-series for:
- **E2.1** Sub-second cross-correlations between cell pairs at fine temporal lags
- **E2.2** Granger causality on within-trial 10 Hz traces
- **E2.3** Directed connection motifs across cell-type pairs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E2.1  Sub-second cross-correlations from 10 Hz ΔF/F traces
# ══════════════════════════════════════════════════════════════════════

# Load from the same mouse used in E1
pk = load_zarr_10hz(mouse_pick)
dff_10hz = pk['dff']           # (n_trials, 41, n_cells)
time_rel_e2 = pk['time_rel']
ti_10hz = pk['trial_info']
uids_10hz = pk['unique_ids']

# Match cells to the subsampled set from E1 (or re-subsample)
obs_mouse_e2 = obs[obs['mouse_id'] == mouse_pick]
cell_map_e2 = {}
for ci, uid in enumerate(uids_10hz):
    m = obs_mouse_e2[obs_mouse_e2['unique_id'] == uid]
    if len(m) > 0:
        cell_map_e2[ci] = m.iloc[0]['subclass_name']

matched_e2 = sorted(cell_map_e2.keys())
sc_e2 = np.array([cell_map_e2[ci] for ci in matched_e2])

# Subsample for tractability
np.random.seed(42)
n_sub_e2 = min(80, len(matched_e2))
sub_idx = np.sort(np.random.choice(len(matched_e2), n_sub_e2, replace=False))
sub_cells = [matched_e2[i] for i in sub_idx]
sub_sc = sc_e2[sub_idx]

# Use stimulus period from non-gray trials
non_gray_e2 = ~ti_10hz['gray_screen'].astype(bool)
grating_idx_e2 = np.where(non_gray_e2.values)[0]
stim_mask_e2 = (time_rel_e2 >= 0) & (time_rel_e2 <= 2.0)
stim_tp_e2 = np.where(stim_mask_e2)[0]

# Concatenate stimulus-period traces across grating trials → pseudo-continuous
n_gt = len(grating_idx_e2)
dff_concat = np.zeros((n_gt * len(stim_tp_e2), n_sub_e2))
for ci_sub, ci in enumerate(sub_cells):
    traces = dff_10hz[grating_idx_e2][:, stim_tp_e2, ci]
    dff_concat[:, ci_sub] = traces.ravel()

print(f"Concatenated traces: {dff_concat.shape} (timepoints × cells) at 10 Hz")
print(f"  = {n_gt} trials × {len(stim_tp_e2)} timepoints/trial = {dff_concat.shape[0]} total")

# ── Compute pairwise cross-correlation at sub-second lags ──
max_lag_e2 = 5  # ±5 samples = ±500 ms
lags_e2 = np.arange(-max_lag_e2, max_lag_e2 + 1) * 0.1

# Compute for all pairs
n_pairs_e2 = n_sub_e2 * (n_sub_e2 - 1) // 2
xcorr_matrix = np.zeros((n_pairs_e2, len(lags_e2)))
pair_info = []
pi = 0
for i in range(n_sub_e2):
    for j in range(i+1, n_sub_e2):
        xcorr_matrix[pi] = xcorr_pair(dff_concat[:, i], dff_concat[:, j], max_lag_e2)
        pair_info.append({'i': i, 'j': j, 'sc_i': sub_sc[i], 'sc_j': sub_sc[j]})
        pi += 1

print(f"Cross-correlations computed for {n_pairs_e2} pairs")

# ── Visualization: mean CCG by pair type ──
pair_df = pd.DataFrame(pair_info)
pair_df['pair_type'] = pair_df.apply(
    lambda r: f"{SUBCLASS_SHORT.get(r['sc_i'], '?')}–{SUBCLASS_SHORT.get(r['sc_j'], '?')}", axis=1)

# Group: E-E, E-I, I-I
def ei_category(sc_i, sc_j):
    ei = 'Glut' in sc_i
    ej = 'Glut' in sc_j
    if ei and ej: return 'E–E'
    if not ei and not ej: return 'I–I'
    return 'E–I'

pair_df['ei_cat'] = [ei_category(r['sc_i'], r['sc_j']) for _, r in pair_df.iterrows()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean CCG by E-E, E-I, I-I
ax = axes[0]
for cat, color in [('E–E', 'green'), ('E–I', 'orange'), ('I–I', 'red')]:
    mask = pair_df['ei_cat'] == cat
    if mask.sum() == 0:
        continue
    mean_cc = np.mean(xcorr_matrix[mask.values], axis=0)
    sem_cc = np.std(xcorr_matrix[mask.values], axis=0) / np.sqrt(mask.sum())
    ax.fill_between(lags_e2, mean_cc - sem_cc, mean_cc + sem_cc, alpha=0.2, color=color)
    ax.plot(lags_e2, mean_cc, color=color, linewidth=2, label=f'{cat} (n={mask.sum()})')
ax.axvline(0, color='k', ls='--', alpha=0.4)
ax.set_xlabel('Lag (s)')
ax.set_ylabel('Correlation')
ax.set_title('E2: Cross-Correlogram by E/I Category', fontweight='bold')
ax.legend(fontsize=9)

# Peak lag distribution
ax = axes[1]
peak_lags_e2 = lags_e2[np.argmax(xcorr_matrix, axis=1)]
for cat, color in [('E–E', 'green'), ('E–I', 'orange'), ('I–I', 'red')]:
    mask = pair_df['ei_cat'] == cat
    ax.hist(peak_lags_e2[mask.values], bins=len(lags_e2), alpha=0.5, color=color,
            label=cat, density=True)
ax.set_xlabel('Peak Lag (s)')
ax.set_ylabel('Density')
ax.set_title('E2: Peak CCG Lag Distribution', fontweight='bold')
ax.legend(fontsize=9)

# Zero-lag vs peak correlation
ax = axes[2]
zero_lag_idx = max_lag_e2
zero_corr = xcorr_matrix[:, zero_lag_idx]
peak_corr_e2 = np.max(xcorr_matrix, axis=1)
ax.scatter(zero_corr, peak_corr_e2, alpha=0.3, s=10, c='steelblue')
ax.plot([0, 0.5], [0, 0.5], 'k--', alpha=0.3)
ax.set_xlabel('Zero-Lag Correlation')
ax.set_ylabel('Peak Correlation')
ax.set_title('E2: Zero-Lag vs Peak Correlation', fontweight='bold')

plt.suptitle('E2: Sub-Second Cross-Correlations (10 Hz)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E2.2  Granger Causality on 10 Hz traces + directed motif analysis
# ══════════════════════════════════════════════════════════════════════

# Use concatenated 10 Hz traces (already computed above)
# Further subsample for GC (computationally expensive)
n_gc = min(50, n_sub_e2)
gc_cell_idx = np.sort(np.random.choice(n_sub_e2, n_gc, replace=False))
gc_sc = sub_sc[gc_cell_idx]

print(f"Computing pairwise Granger causality for {n_gc} cells...")
gc_results = []
pair_count = 0
n_gc_pairs = n_gc * (n_gc - 1) // 2

for i in range(n_gc):
    for j in range(i+1, n_gc):
        xi = dff_concat[:, gc_cell_idx[i]]
        xj = dff_concat[:, gc_cell_idx[j]]
        # Remove NaN segments
        valid = ~np.isnan(xi) & ~np.isnan(xj)
        if valid.sum() < 50:
            continue
        res = pairwise_granger(xi[valid], xj[valid], max_lag=3)
        gc_results.append({
            'i': i, 'j': j,
            'sc_i': gc_sc[i], 'sc_j': gc_sc[j],
            'p_j_to_i': res['y_to_x_p'],
            'p_i_to_j': res['x_to_y_p'],
        })
        pair_count += 1
        if pair_count % 200 == 0:
            print(f"  {pair_count}/{n_gc_pairs} pairs...")

gc_df = pd.DataFrame(gc_results)
gc_df['sig_j_to_i'] = gc_df['p_j_to_i'] < 0.01
gc_df['sig_i_to_j'] = gc_df['p_i_to_j'] < 0.01
gc_df['sig_any'] = gc_df['sig_j_to_i'] | gc_df['sig_i_to_j']

total_edges = gc_df['sig_j_to_i'].sum() + gc_df['sig_i_to_j'].sum()
print(f"\nSignificant directed edges: {total_edges} out of {2 * len(gc_df)} possible "
      f"({total_edges/(2*len(gc_df))*100:.1f}%)")

# ── Directed connection probability matrix: subclass × subclass ──
present_sc_gc = [s for s in present_subclasses if s in gc_sc]
n_sgc = len(present_sc_gc)
directed_prob = np.zeros((n_sgc, n_sgc))
directed_count = np.zeros((n_sgc, n_sgc))

for _, row in gc_df.iterrows():
    if row['sc_i'] not in present_sc_gc or row['sc_j'] not in present_sc_gc:
        continue
    si = present_sc_gc.index(row['sc_i'])
    sj = present_sc_gc.index(row['sc_j'])
    directed_count[si, sj] += 1
    directed_count[sj, si] += 1
    if row['sig_i_to_j']:
        directed_prob[si, sj] += 1
    if row['sig_j_to_i']:
        directed_prob[sj, si] += 1

directed_frac = directed_prob / np.maximum(directed_count, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
s_labels_gc = [SUBCLASS_SHORT.get(s, s) for s in present_sc_gc]

ax = axes[0]
sns.heatmap(directed_frac, annot=True, fmt='.3f', xticklabels=s_labels_gc, yticklabels=s_labels_gc,
            cmap='Reds', ax=ax, vmin=0)
ax.set_xlabel('To')
ax.set_ylabel('From')
ax.set_title('E2: Directed GC Connection Probability\n(from row → to column)', fontweight='bold')

# Asymmetry
ax = axes[1]
asym = np.zeros_like(directed_frac)
for i in range(n_sgc):
    for j in range(n_sgc):
        denom = directed_frac[i, j] + directed_frac[j, i]
        if denom > 0:
            asym[i, j] = (directed_frac[i, j] - directed_frac[j, i]) / denom

sns.heatmap(asym, annot=True, fmt='.2f', xticklabels=s_labels_gc, yticklabels=s_labels_gc,
            cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_xlabel('To')
ax.set_ylabel('From')
ax.set_title('E2: Connectivity Asymmetry Index\n(+ = net flow From→To)', fontweight='bold')

plt.suptitle('E2: Granger Causality on 10 Hz ΔF/F', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── GC-significant connection probability vs distance ──
# Re-derive coordinates for the GC-subsampled cells
gc_cell_global = [matched_e2[gc_cell_idx[i]] for i in range(n_gc)]
gc_uids = [uids_10hz[ci] for ci in gc_cell_global]
gc_coords = np.zeros((n_gc, 3))
for i, uid in enumerate(gc_uids):
    row = obs_mouse_e2[obs_mouse_e2['unique_id'] == uid]
    if len(row) > 0:
        gc_coords[i] = [row.iloc[0]['x_loc'], row.iloc[0]['y_loc'], row.iloc[0].get('z_loc', 0)]

gc_df['dist'] = [np.linalg.norm(gc_coords[r['i']] - gc_coords[r['j']]) for _, r in gc_df.iterrows()]

fig, ax = plt.subplots(figsize=(8, 5))
dist_bins = np.arange(0, 400, 50)
dist_centers = (dist_bins[:-1] + dist_bins[1:]) / 2
prob_vs_dist = []
for b in range(len(dist_bins)-1):
    bm = (gc_df['dist'] >= dist_bins[b]) & (gc_df['dist'] < dist_bins[b+1])
    if bm.sum() > 5:
        prob_vs_dist.append(gc_df.loc[bm, 'sig_any'].mean())
    else:
        prob_vs_dist.append(np.nan)

ax.plot(dist_centers, prob_vs_dist, 'ko-', linewidth=2, markersize=6)
ax.set_xlabel('Pairwise Distance (µm)')
ax.set_ylabel('P(significant GC connection)')
ax.set_title('E2: Directed Connection Probability vs Distance', fontweight='bold')
plt.tight_layout()
plt.show()

## E3: Population Coupling

Measure how strongly each cell's activity correlates with the population mean, split by running state and subclass. Relate coupling to spatial position and gene expression.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E3.1  Population coupling index per cell
# ══════════════════════════════════════════════════════════════════════

# For each mouse, compute coupling as correlation between each cell's
# trial responses and the population mean (excluding that cell)

run_mask_t = var['is_running'].values.astype(bool)
coupling_all = np.full(adata.n_obs, np.nan)
coupling_run = np.full(adata.n_obs, np.nan)
coupling_stat = np.full(adata.n_obs, np.nan)

for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_X = X[m_mask]
    n_m = m_mask.sum()
    m_indices = np.where(m_mask)[0]
    
    # Population mean (excluding self) — efficiently computed
    pop_sum = np.nansum(m_X, axis=0)
    
    for i in range(n_m):
        # Population mean excluding this cell
        pop_exc = (pop_sum - m_X[i]) / (n_m - 1)
        
        # Overall coupling
        valid = ~np.isnan(m_X[i]) & ~np.isnan(pop_exc)
        if valid.sum() > 20:
            coupling_all[m_indices[i]], _ = pearsonr(m_X[i, valid], pop_exc[valid])
        
        # Running trials only
        run_valid = valid & run_mask_t
        if run_valid.sum() > 10:
            coupling_run[m_indices[i]], _ = pearsonr(m_X[i, run_valid], pop_exc[run_valid])
        
        # Stationary trials only
        stat_valid = valid & ~run_mask_t
        if stat_valid.sum() > 10:
            coupling_stat[m_indices[i]], _ = pearsonr(m_X[i, stat_valid], pop_exc[stat_valid])

coupling_df = pd.DataFrame({
    'coupling': coupling_all,
    'coupling_run': coupling_run,
    'coupling_stat': coupling_stat,
    'subclass': obs['subclass_name'].values,
    'subclass_short': obs['subclass_name'].map(SUBCLASS_SHORT).values,
    'x': coords[:, 0],
    'y': coords[:, 1],
    'z': coords[:, 2],
    'mouse_id': obs['mouse_id'].values,
})

print("Population coupling by subclass:")
print(coupling_df.groupby('subclass_short')['coupling'].describe().round(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E3.2  Visualization: Coupling by subclass, running state, and space
# ══════════════════════════════════════════════════════════════════════

short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Violin: coupling by subclass
ax = axes[0, 0]
sns.violinplot(data=coupling_df[coupling_df['subclass'].isin(present_subclasses)],
               x='subclass_short', y='coupling', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.4)
ax.set_title('E3: Population Coupling by Subclass', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Coupling (Pearson r)')
ax.tick_params(axis='x', rotation=45)

# Running vs Stationary coupling
ax = axes[0, 1]
melt = coupling_df[coupling_df['subclass'].isin(present_subclasses)].melt(
    id_vars=['subclass_short'], value_vars=['coupling_run', 'coupling_stat'],
    var_name='state', value_name='coupling_val')
melt['state'] = melt['state'].map({'coupling_run': 'Running', 'coupling_stat': 'Stationary'})
sns.boxplot(data=melt, x='subclass_short', y='coupling_val', hue='state',
            order=short_order, palette={'Running': 'red', 'Stationary': 'blue'}, ax=ax)
ax.set_title('E3: Coupling by Running State', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Coupling')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=45)
ax.axhline(0, color='k', ls='--', alpha=0.4)

# Coupling vs depth
ax = axes[1, 0]
for sc in present_subclasses:
    mask = coupling_df['subclass'] == sc
    valid = mask & coupling_df['coupling'].notna()
    ax.scatter(coupling_df.loc[valid, 'z'], coupling_df.loc[valid, 'coupling'],
               alpha=0.3, s=10, c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('Population Coupling')
ax.set_title('E3: Coupling vs Cortical Depth', fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.axhline(0, color='k', ls='--', alpha=0.4)

# Spatial map of coupling
ax = axes[1, 1]
valid_c = coupling_df['coupling'].notna()
sc = ax.scatter(coupling_df.loc[valid_c, 'x'], coupling_df.loc[valid_c, 'y'],
                c=coupling_df.loc[valid_c, 'coupling'], cmap='coolwarm', s=10, alpha=0.6,
                vmin=-0.3, vmax=0.3)
plt.colorbar(sc, ax=ax, label='Coupling')
ax.set_xlabel('X (µm)')
ax.set_ylabel('Y (µm)')
ax.set_title('E3: Spatial Map of Population Coupling', fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# ── Kruskal-Wallis ──
groups = [coupling_df.loc[coupling_df['subclass'] == s, 'coupling'].dropna().values
          for s in present_subclasses]
groups = [g for g in groups if len(g) >= 3]
stat, p = kruskal(*groups)
print(f"Coupling Kruskal-Wallis: H={stat:.2f}, p={p:.2e}")

# ── Coupling vs depth (Spearman) per subclass ──
print("\nCoupling vs depth:")
for sc in present_subclasses:
    mask = (coupling_df['subclass'] == sc) & coupling_df['coupling'].notna()
    if mask.sum() >= 10:
        r, p = spearmanr(coupling_df.loc[mask, 'z'], coupling_df.loc[mask, 'coupling'])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: ρ={r:.3f}, p={p:.2e}")

## E4: Spontaneous Activity Assemblies (Grey-Screen Epochs)

Using the ~360 s spontaneous activity (GreyScreen) recorded at 10 Hz from zarr stores, detect neural assemblies via NMF, characterize their cell-type composition, and examine running-state modulation.

## E5: Catch-Trial Expectation Signals

Analyse catch (blank) trials interleaved among gratings to detect cell-type-specific expectation or omission signals. 27 catch trials per session, 51 timepoints (−1 to +4 s).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E4  Spontaneous Activity Assemblies from Grey-Screen Data (Zarr)
# ══════════════════════════════════════════════════════════════════════
import zarr
from sklearn.decomposition import NMF

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']

# ── Load spontaneous activity from zarr ──
assembly_results = {}
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    
    # Use session_1, long grey-screen epoch (index 1, ~360 s)
    gs = z['ophys/drifting_gratings/session_1/stim_aligned_dff/GreyScreen']
    dff_spont = gs['dff'][1]          # (3624, n_cells)
    running_spont = gs['running'][1]  # (3624, 2) — speed, acceleration
    time_spont = gs['time_relative'][:]
    
    # Cell-type labels
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    n_cells = dff_spont.shape[1]
    
    # Clip negative values for NMF (shift to non-negative)
    dff_shifted = dff_spont - dff_spont.min(axis=0, keepdims=True)
    
    # NMF assembly detection
    n_assemblies = 10
    nmf = NMF(n_components=n_assemblies, init='nndsvda', random_state=42, max_iter=500)
    W = nmf.fit_transform(dff_shifted)  # (timepoints, n_assemblies) — activation timecourses
    H = nmf.components_                  # (n_assemblies, n_cells) — cell weights
    
    # Assembly cell-type composition
    composition = {}
    for a in range(n_assemblies):
        top_cells = np.argsort(H[a])[-int(n_cells * 0.1):]  # top 10% contributing cells
        types_in_assembly = subclass_names[top_cells]
        unique, counts = np.unique(types_in_assembly, return_counts=True)
        composition[a] = dict(zip(unique, counts))
    
    # Running–assembly correlation
    run_speed = running_spont[:, 0]
    assembly_run_corr = np.array([pearsonr(W[:, a], run_speed)[0] for a in range(n_assemblies)])
    
    assembly_results[mouse_id] = {
        'W': W, 'H': H, 'composition': composition,
        'assembly_run_corr': assembly_run_corr,
        'subclass_names': subclass_names, 'run_speed': run_speed,
        'time': time_spont, 'reconstruction_error': nmf.reconstruction_err_
    }
    print(f"Mouse {mouse_id}: {n_cells} cells, {n_assemblies} assemblies, "
          f"recon error={nmf.reconstruction_err_:.1f}")

# ── Visualization ──
fig, axes = plt.subplots(len(MOUSE_IDS), 3, figsize=(20, 4 * len(MOUSE_IDS)))
if len(MOUSE_IDS) == 1:
    axes = axes[np.newaxis, :]

for i, mouse_id in enumerate(MOUSE_IDS):
    res = assembly_results[mouse_id]
    
    # Assembly activation timecourse (first 500 timepoints = 50 s)
    ax = axes[i, 0]
    t_plot = res['time'][:500]
    for a in range(min(5, res['W'].shape[1])):
        ax.plot(t_plot, res['W'][:500, a], alpha=0.7, label=f'Asm {a}')
    ax2 = ax.twinx()
    ax2.plot(t_plot, res['run_speed'][:500], 'k-', alpha=0.3, lw=0.5)
    ax2.set_ylabel('Run speed', fontsize=8)
    ax.set_title(f'Mouse {mouse_id}: Assembly Activations', fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Activation')
    ax.legend(fontsize=6, loc='upper right')
    
    # Assembly–running correlation
    ax = axes[i, 1]
    colors = ['red' if c > 0.1 else 'blue' if c < -0.1 else 'gray'
              for c in res['assembly_run_corr']]
    ax.bar(range(len(res['assembly_run_corr'])), res['assembly_run_corr'], color=colors)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    ax.set_xlabel('Assembly')
    ax.set_ylabel('Correlation with running')
    ax.set_title(f'Mouse {mouse_id}: Assembly–Running Coupling', fontweight='bold')
    
    # Assembly composition (stacked bar)
    ax = axes[i, 2]
    all_types = sorted(set(t for comp in res['composition'].values() for t in comp))
    type_colors = [SUBCLASS_COLORS.get(t, '#cccccc') for t in all_types]
    bottoms = np.zeros(len(res['composition']))
    for j, t in enumerate(all_types):
        vals = [res['composition'][a].get(t, 0) for a in range(len(res['composition']))]
        ax.bar(range(len(vals)), vals, bottom=bottoms, color=type_colors[j],
               label=SUBCLASS_SHORT.get(t, t[-10:]))
        bottoms += vals
    ax.set_xlabel('Assembly')
    ax.set_ylabel('# top-10% cells')
    ax.set_title(f'Mouse {mouse_id}: Assembly Composition', fontweight='bold')
    ax.legend(fontsize=6, loc='upper right')

plt.tight_layout()
plt.show()

# ── Participation breadth by subclass ──
print("\n=== Cell participation breadth (# assemblies with above-median weight) ===")
for mouse_id in MOUSE_IDS:
    res = assembly_results[mouse_id]
    H = res['H']
    medians = np.median(H, axis=1, keepdims=True)
    participation = (H > medians).sum(axis=0)  # per cell, how many assemblies
    for sc in present_subclasses:
        mask = res['subclass_names'] == sc
        if mask.sum() > 0:
            print(f"  {mouse_id} {SUBCLASS_SHORT[sc]:10s}: "
                  f"mean={participation[mask].mean():.1f}, n={mask.sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E5  Catch-Trial Expectation / Omission Signals (Zarr)
# ══════════════════════════════════════════════════════════════════════

catch_results = []
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    
    for sess in ['session_1', 'session_2', 'session_3']:
        catch = z[f'ophys/drifting_gratings/{sess}/stim_aligned_dff/Catch']
        dff_catch = catch['dff'][:]         # (27, 51, n_cells)
        time_catch = catch['time_relative'][:]  # (51,)
        running_catch = catch['running'][:]  # (27, 51, 2)
        
        # Trial info: which context block the catch occurred in
        stim_names = catch['trial_info/stim_name'][:]
        
        # Baseline: mean of −1 to 0 s (first 10 timepoints)
        baseline_idx = time_catch < 0
        stim_idx = (time_catch >= 0) & (time_catch <= 2.0)
        
        baseline_mean = dff_catch[:, baseline_idx, :].mean(axis=(0, 1))  # (n_cells,)
        stim_mean = dff_catch[:, stim_idx, :].mean(axis=(0, 1))          # (n_cells,)
        
        # Catch response index per cell
        catch_response = stim_mean - baseline_mean
        
        for ci, sc in enumerate(subclass_names):
            catch_results.append({
                'mouse_id': mouse_id, 'session': sess,
                'subclass': sc, 'catch_response': catch_response[ci],
                'cell_idx': ci
            })

catch_df = pd.DataFrame(catch_results)
catch_df['subclass_short'] = catch_df['subclass'].map(SUBCLASS_SHORT)

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Catch response distributions by subclass
ax = axes[0]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}
catch_avg = catch_df.groupby(['mouse_id', 'cell_idx', 'subclass_short'])['catch_response'].mean().reset_index()
sns.violinplot(data=catch_avg[catch_avg['subclass_short'].isin(short_order)],
               x='subclass_short', y='catch_response', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.4)
ax.set_title('E5: Catch-Trial Response by Subclass', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('ΔF/F (stim − baseline)')
ax.tick_params(axis='x', rotation=45)

# 2. Mean PSTH for catch trials by subclass (session 1, mouse 778174)
ax = axes[1]
z = zarr.open(f'{ZARR_DIR}/778174_multimodal_data.zarr', 'r')
dff_c = z['ophys/drifting_gratings/session_1/stim_aligned_dff/Catch/dff'][:]
t_c = z['ophys/drifting_gratings/session_1/stim_aligned_dff/Catch/time_relative'][:]
sc_names = z['transcriptomics/cell_type/subclass_name'][:]
for sc in present_subclasses:
    mask = sc_names == sc
    if mask.sum() > 0:
        psth = dff_c[:, :, mask].mean(axis=(0, 2))
        ax.plot(t_c, psth, color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc], lw=1.5)
ax.axvline(0, color='k', ls='--', alpha=0.3, label='Stim onset')
ax.axhspan(-0.01, 0.01, color='gray', alpha=0.1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Mean ΔF/F')
ax.set_title('E5: Catch-Trial PSTHs (778174, Session 1)', fontweight='bold')
ax.legend(fontsize=7)

# 3. Fraction of cells with significant catch responses
ax = axes[2]
sig_fractions = {}
for sc in present_subclasses:
    sc_data = catch_avg[catch_avg['subclass_short'] == SUBCLASS_SHORT[sc]]['catch_response']
    if len(sc_data) > 5:
        # Fraction significantly positive or negative (> 2 SD from 0)
        threshold = 2 * sc_data.std()
        frac_pos = (sc_data > threshold).mean()
        frac_neg = (sc_data < -threshold).mean()
        sig_fractions[SUBCLASS_SHORT[sc]] = {'positive': frac_pos, 'negative': frac_neg}

if sig_fractions:
    types = list(sig_fractions.keys())
    pos_vals = [sig_fractions[t]['positive'] for t in types]
    neg_vals = [sig_fractions[t]['negative'] for t in types]
    x = np.arange(len(types))
    ax.bar(x - 0.15, pos_vals, 0.3, color='firebrick', label='Activated', alpha=0.8)
    ax.bar(x + 0.15, neg_vals, 0.3, color='steelblue', label='Suppressed', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=45)
    ax.set_ylabel('Fraction of cells')
    ax.set_title('E5: Catch-Responsive Cells', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Statistics ──
print("Catch-trial response by subclass (Kruskal-Wallis):")
groups = [catch_avg.loc[catch_avg['subclass_short'] == SUBCLASS_SHORT[s], 'catch_response'].dropna().values
          for s in present_subclasses if (catch_avg['subclass_short'] == SUBCLASS_SHORT[s]).sum() >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f"  H={stat:.2f}, p={p:.2e}")